# Tercera Entrega – Modelado y Evaluación
**Proyecto:** Predicción de Duración de Partidos de Tenis  
**Fecha:** 22/10/2025  
**Alumno:** Juan Ignacio Barranco Bastan

---

## Justificación del cambio de objetivo

En la **segunda entrega (EDA)** se realizó un análisis exploratorio exhaustivo del dataset `matches_cleaned.csv`. Según el feedback del docente:

> **El objetivo original de "predecir el campeón del torneo" no es viable con los datos disponibles**, ya que:
> - No contamos con información a priori de los resultados finales del torneo
> - Los datos describen partidos individuales, no torneos completos
> - La mayoría de las variables relevantes del EDA están relacionadas con la **duración y desarrollo de los partidos**

Por lo tanto, **redefinimos el objetivo predictivo** como:

> **Predecir la duración del partido (en minutos)** utilizando características conocidas **antes del inicio del encuentro** (jugadores, ranking, superficie, ronda, etc.)

Este objetivo es:
- ✅ **Factible** con los datos disponibles
- ✅ **Relevante** para análisis deportivos, logística de torneos y broadcasting
- ✅ **Justificado** por el EDA previo que mostró alta variabilidad en duraciones y correlaciones con `best_of`, `surface`, `is_grand_slam`

### Tipo de problema

**Regresión**: la variable objetivo `minutes` es continua (duración en minutos).

---

## 1. Importación de librerías y carga de datos

In [ ]:
# Librerías básicas
import sys, platform
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de visualización
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Scikit-learn: preprocesamiento
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# Scikit-learn: modelos de regresión
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor

# Métricas de regresión
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Para transformadores personalizados
from sklearn.base import BaseEstimator, TransformerMixin

import warnings
warnings.filterwarnings('ignore')

print("Python:", sys.version.split()[0], "| OS:", platform.system(), platform.release())
print(f"pandas: {pd.__version__} | numpy: {np.__version__} | sklearn: {__import__('sklearn').__version__}")

In [ ]:
# Cargar dataset limpio desde la segunda entrega
df = pd.read_csv('entrega2proy_EDA/matches_cleaned.csv')

print(f"Dataset cargado: {df.shape[0]} partidos, {df.shape[1]} variables")
df.head()

## 2. Definición del problema predictivo

### Variable objetivo

**`minutes`**: Duración total del partido en minutos

### Features (variables predictoras)

Seleccionamos variables conocidas **antes** del partido:

- **Información del torneo**: `tourney_level`, `surface`, `round`, `best_of`
- **Características de los jugadores**: `winner_rank`, `loser_rank`, `winner_age`, `loser_age`, `winner_hand`, `loser_hand`, `winner_ht`, `loser_ht`

**Excluimos** variables que solo se conocen **después** del partido:
- Estadísticas del partido: `w_ace`, `l_ace`, `w_df`, `l_df`, etc.
- Resultado final: `score`
- Nombres de jugadores: `winner_name`, `loser_name` (para evitar data leakage)

### Estrategia de validación

- **División train/test**: 80% / 20% con estratificación por `best_of`
- **Validación cruzada**: 5-fold CV durante ajuste de hiperparámetros

In [ ]:
# Variable objetivo
target = 'minutes'

# Features seleccionadas (conocidas antes del partido)
features = [
    'tourney_level', 'surface', 'round', 'best_of',
    'winner_rank', 'loser_rank', 
    'winner_age', 'loser_age',
    'winner_hand', 'loser_hand',
    'winner_ht', 'loser_ht'
]

# Verificar que las columnas existen
missing_cols = set(features + [target]) - set(df.columns)
if missing_cols:
    print(f"⚠️ Columnas faltantes: {missing_cols}")
else:
    print("✅ Todas las columnas están disponibles")

# Crear subset con features + target
df_model = df[features + [target]].copy()

# Eliminar filas con target faltante o minutos <= 0
df_model = df_model.dropna(subset=[target])
df_model = df_model[df_model[target] > 0]

print(f"\nDataset para modelado: {df_model.shape[0]} partidos")
print(f"\nDistribución de la variable objetivo:")
print(df_model[target].describe())

## 3. Análisis exploratorio de la variable objetivo

In [ ]:
# Distribución de la duración de partidos
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
axes[0].hist(df_model[target], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Duración (minutos)')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de duración de partidos')
axes[0].axvline(df_model[target].median(), color='red', linestyle='--', 
                label=f'Mediana: {df_model[target].median():.0f} min')
axes[0].legend()

# Boxplot
axes[1].boxplot(df_model[target], vert=True)
axes[1].set_ylabel('Duración (minutos)')
axes[1].set_title('Boxplot de duración')

plt.tight_layout()
plt.show()

print(f"\nAsimetría (skewness): {df_model[target].skew():.2f}")
print(f"Curtosis (kurtosis): {df_model[target].kurtosis():.2f}")

**Observaciones:**
- La distribución de duración tiene asimetría positiva (partidos largos como outliers)
- Presencia de outliers (partidos muy largos >300 min, identificados en el EDA)
- Esto sugiere que modelos robustos (Random Forest, Gradient Boosting) pueden funcionar mejor que regresión lineal simple

## 4. División del dataset: Entrenamiento y Test

Aplicamos `train_test_split` con **estratificación por `best_of`** para mantener proporciones de partidos a 3 y 5 sets.

In [ ]:
# Separar features y target
X = df_model[features]
y = df_model[target]

# División 80% train / 20% test, con estratificación por best_of
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=X['best_of']  # Mantener proporción de partidos a 3 y 5 sets
)

print(f"Train: {X_train.shape[0]} partidos")
print(f"Test: {X_test.shape[0]} partidos")
print(f"\nProporción best_of en train:")
print(X_train['best_of'].value_counts(normalize=True))
print(f"\nProporción best_of en test:")
print(X_test['best_of'].value_counts(normalize=True))

## 5. Construcción del Pipeline de preprocesamiento

Siguiendo las mejores prácticas de scikit-learn (inspiradas en los notebooks del docente sobre aprendizaje supervisado), construimos un pipeline modular que:

1. **Identifica tipos de columnas** (numéricas vs categóricas)
2. **Imputa valores faltantes**
3. **Escala variables numéricas** (StandardScaler)
4. **Codifica variables categóricas** (OneHotEncoder)
5. **Integra todo en un ColumnTransformer**

Este diseño evita **data leakage** porque las transformaciones se ajustan solo con datos de entrenamiento.

In [ ]:
# Identificar columnas numéricas y categóricas
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Features numéricas ({len(numeric_features)}): {numeric_features}")
print(f"Features categóricas ({len(categorical_features)}): {categorical_features}")

# Pipeline para variables numéricas: imputación + escalado
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline para variables categóricas: imputación + one-hot encoding
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

# ColumnTransformer: aplicar transformaciones específicas a cada tipo
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

print("\n✅ Preprocessor creado correctamente")

## 6. Comparación de modelos de regresión

Evaluamos **tres modelos diferentes** (requisito de la consigna):

1. **Ridge Regression**: Regresión lineal con regularización L2 (control de overfitting)
2. **Random Forest Regressor**: Ensamble de árboles de decisión (captura no linealidades)
3. **Gradient Boosting Regressor**: Ensamble secuencial de árboles (state-of-the-art para tabular)

Cada modelo se entrena dentro de un **pipeline completo** que incluye preprocesamiento.

In [ ]:
# Pipeline 1: Ridge Regression
pipeline_ridge = Pipeline([
    ('preprocessor', preprocessor),
    ('model', Ridge(random_state=42))
])

# Pipeline 2: Random Forest
pipeline_rf = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])

# Pipeline 3: Gradient Boosting
pipeline_gb = Pipeline([
    ('preprocessor', preprocessor),
    ('model', GradientBoostingRegressor(n_estimators=100, random_state=42))
])

# Entrenar los tres modelos
print("Entrenando modelos...\n")

print("⏳ Entrenando Ridge Regression...")
pipeline_ridge.fit(X_train, y_train)
print("✅ Ridge Regression entrenado")

print("\n⏳ Entrenando Random Forest...")
pipeline_rf.fit(X_train, y_train)
print("✅ Random Forest entrenado")

print("\n⏳ Entrenando Gradient Boosting...")
pipeline_gb.fit(X_train, y_train)
print("✅ Gradient Boosting entrenado")

## 7. Evaluación de modelos

Utilizamos métricas estándar de regresión:

- **RMSE (Root Mean Squared Error)**: Error cuadrático medio (penaliza errores grandes)
- **MAE (Mean Absolute Error)**: Error absoluto medio (más robusto a outliers)
- **R² (Coeficiente de determinación)**: Proporción de varianza explicada (0-1, mayor es mejor)

In [ ]:
def evaluar_modelo(nombre, pipeline, X_train, y_train, X_test, y_test):
    """Evalúa un modelo de regresión en train y test"""
    y_train_pred = pipeline.predict(X_train)
    y_test_pred = pipeline.predict(X_test)
    
    # Métricas en train
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    train_mae = mean_absolute_error(y_train, y_train_pred)
    train_r2 = r2_score(y_train, y_train_pred)
    
    # Métricas en test
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    test_mae = mean_absolute_error(y_test, y_test_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    
    print(f"\n{'='*50}")
    print(f"🔍 {nombre}")
    print(f"{'='*50}")
    print(f"\n📊 Train:")
    print(f"  RMSE: {train_rmse:.2f} min")
    print(f"  MAE:  {train_mae:.2f} min")
    print(f"  R²:   {train_r2:.4f}")
    print(f"\n📊 Test:")
    print(f"  RMSE: {test_rmse:.2f} min")
    print(f"  MAE:  {test_mae:.2f} min")
    print(f"  R²:   {test_r2:.4f}")
    
    # Diagnóstico de overfitting
    diff_r2 = train_r2 - test_r2
    if diff_r2 > 0.1:
        print(f"\n⚠️ Posible overfitting (diferencia R²: {diff_r2:.4f})")
    elif test_r2 < 0.3:
        print(f"\n⚠️ Posible underfitting (R² test bajo: {test_r2:.4f})")
    else:
        print(f"\n✅ Generalización adecuada")
    
    return {
        'train_rmse': train_rmse, 'train_mae': train_mae, 'train_r2': train_r2,
        'test_rmse': test_rmse, 'test_mae': test_mae, 'test_r2': test_r2
    }

# Evaluar los tres modelos
results = {}
results['Ridge'] = evaluar_modelo('Ridge Regression', pipeline_ridge, X_train, y_train, X_test, y_test)
results['RandomForest'] = evaluar_modelo('Random Forest', pipeline_rf, X_train, y_train, X_test, y_test)
results['GradientBoosting'] = evaluar_modelo('Gradient Boosting', pipeline_gb, X_train, y_train, X_test, y_test)

In [ ]:
# Comparación visual de modelos
df_results = pd.DataFrame(results).T

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# RMSE
df_results[['train_rmse', 'test_rmse']].plot(kind='bar', ax=axes[0], color=['skyblue', 'salmon'])
axes[0].set_title('RMSE (menor es mejor)')
axes[0].set_ylabel('Error (minutos)')
axes[0].legend(['Train', 'Test'])
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')

# MAE
df_results[['train_mae', 'test_mae']].plot(kind='bar', ax=axes[1], color=['skyblue', 'salmon'])
axes[1].set_title('MAE (menor es mejor)')
axes[1].set_ylabel('Error (minutos)')
axes[1].legend(['Train', 'Test'])
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')

# R²
df_results[['train_r2', 'test_r2']].plot(kind='bar', ax=axes[2], color=['skyblue', 'salmon'])
axes[2].set_title('R² (mayor es mejor)')
axes[2].set_ylabel('R² score')
axes[2].legend(['Train', 'Test'])
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

## 8. Selección del mejor modelo y justificación

**Criterios de selección:**
1. Menor RMSE en test (error de predicción)
2. Mayor R² en test (capacidad explicativa)
3. Equilibrio entre train y test (evitar overfitting)

**Decisión:**

In [ ]:
# Seleccionar el mejor modelo según RMSE en test
best_model_name = min(results, key=lambda x: results[x]['test_rmse'])
best_model = {'Ridge': pipeline_ridge, 'RandomForest': pipeline_rf, 'GradientBoosting': pipeline_gb}[best_model_name]

print(f"\n{'='*60}")
print(f"🏆 MODELO SELECCIONADO: {best_model_name}")
print(f"{'='*60}")
print(f"\n📈 Mejores métricas en test:")
print(f"  RMSE: {results[best_model_name]['test_rmse']:.2f} min")
print(f"  MAE:  {results[best_model_name]['test_mae']:.2f} min")
print(f"  R²:   {results[best_model_name]['test_r2']:.4f}")
print(f"\n💡 Justificación:")
print(f"   Este modelo presenta el mejor balance entre capacidad predictiva")
print(f"   y generalización, minimizando el error en datos no vistos.")
print(f"   El R² indica que explica ~{results[best_model_name]['test_r2']*100:.1f}% de la varianza en duración.")

## 9. Ajuste de hiperparámetros (GridSearchCV)

Aplicamos búsqueda en grilla sobre el modelo seleccionado para optimizar sus hiperparámetros mediante **validación cruzada 5-fold**.

In [ ]:
# Definir grilla de hiperparámetros según el modelo seleccionado
if best_model_name == 'RandomForest':
    param_grid = {
        'model__n_estimators': [50, 100, 200],
        'model__max_depth': [None, 10, 20, 30],
        'model__min_samples_split': [2, 5, 10]
    }
elif best_model_name == 'GradientBoosting':
    param_grid = {
        'model__n_estimators': [50, 100, 200],
        'model__learning_rate': [0.01, 0.1, 0.2],
        'model__max_depth': [3, 5, 7]
    }
else:  # Ridge
    param_grid = {
        'model__alpha': [0.01, 0.1, 1, 10, 100]
    }

# GridSearchCV con validación cruzada
grid_search = GridSearchCV(
    estimator=best_model,
    param_grid=param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

print(f"\n🔧 Iniciando búsqueda de hiperparámetros para {best_model_name}...\n")
grid_search.fit(X_train, y_train)

print(f"\n✅ Búsqueda completada")
print(f"\n🎯 Mejores hiperparámetros encontrados:")
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")

print(f"\n📊 Mejor RMSE (validación cruzada): {-grid_search.best_score_:.2f} min")

## 10. Evaluación del modelo optimizado

In [ ]:
# Obtener el mejor estimador
best_estimator = grid_search.best_estimator_

# Evaluar en test
y_test_pred_optimized = best_estimator.predict(X_test)

test_rmse_opt = np.sqrt(mean_squared_error(y_test, y_test_pred_optimized))
test_mae_opt = mean_absolute_error(y_test, y_test_pred_optimized)
test_r2_opt = r2_score(y_test, y_test_pred_optimized)

print(f"\n{'='*60}")
print(f"📈 MODELO OPTIMIZADO - Evaluación en Test")
print(f"{'='*60}")
print(f"\n  RMSE: {test_rmse_opt:.2f} min")
print(f"  MAE:  {test_mae_opt:.2f} min")
print(f"  R²:   {test_r2_opt:.4f}")

# Comparar con el modelo base
mejora_rmse = results[best_model_name]['test_rmse'] - test_rmse_opt
mejora_r2 = test_r2_opt - results[best_model_name]['test_r2']

print(f"\n🚀 Mejora respecto al modelo base:")
print(f"  RMSE: {mejora_rmse:+.2f} min ({(mejora_rmse/results[best_model_name]['test_rmse']*100):+.1f}%)")
print(f"  R²:   {mejora_r2:+.4f}")

## 11. Diagnóstico: Overfitting vs Underfitting

In [ ]:
# Predecir en train con el modelo optimizado
y_train_pred_opt = best_estimator.predict(X_train)

train_rmse_opt = np.sqrt(mean_squared_error(y_train, y_train_pred_opt))
train_r2_opt = r2_score(y_train, y_train_pred_opt)

print(f"\n{'='*60}")
print(f"🔬 DIAGNÓSTICO: Overfitting / Underfitting")
print(f"{'='*60}")
print(f"\nRMSE Train: {train_rmse_opt:.2f} min")
print(f"RMSE Test:  {test_rmse_opt:.2f} min")
print(f"Diferencia: {abs(train_rmse_opt - test_rmse_opt):.2f} min")
print(f"\nR² Train:  {train_r2_opt:.4f}")
print(f"R² Test:   {test_r2_opt:.4f}")
print(f"Diferencia: {abs(train_r2_opt - test_r2_opt):.4f}")

# Criterios de diagnóstico
diff_r2 = train_r2_opt - test_r2_opt

if diff_r2 > 0.15:
    print(f"\n⚠️ OVERFITTING DETECTADO")
    print("   El modelo memoriza el training set pero no generaliza bien.")
    print("   Acciones sugeridas:")
    print("   • Aumentar regularización")
    print("   • Reducir complejidad del modelo (max_depth, n_estimators)")
    print("   • Conseguir más datos de entrenamiento")
    print("   • Aplicar early stopping (Gradient Boosting)")
elif test_r2_opt < 0.3:
    print(f"\n⚠️ UNDERFITTING DETECTADO")
    print("   El modelo no captura patrones relevantes en los datos.")
    print("   Acciones sugeridas:")
    print("   • Aumentar complejidad del modelo")
    print("   • Agregar features más informativas")
    print("   • Probar modelos no lineales más complejos")
    print("   • Ingeniería de features (interacciones, transformaciones)")
else:
    print(f"\n✅ GENERALIZACIÓN ADECUADA")
    print("   El modelo tiene buen equilibrio entre sesgo y varianza.")
    print("   La diferencia entre train y test es razonable (~{:.1f}% en R²).".format(diff_r2*100))

## 12. Visualización de predicciones

In [ ]:
# Gráfico de predicciones vs valores reales
fig, ax = plt.subplots(figsize=(8, 8))

ax.scatter(y_test, y_test_pred_optimized, alpha=0.5, edgecolor='k')
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
        'r--', lw=2, label='Predicción perfecta')
ax.set_xlabel('Duración real (minutos)')
ax.set_ylabel('Duración predicha (minutos)')
ax.set_title(f'Predicciones vs Valores Reales\n({best_model_name} optimizado)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Distribución de errores
errores = y_test - y_test_pred_optimized

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(errores, bins=50, edgecolor='black', alpha=0.7)
ax.axvline(0, color='red', linestyle='--', label='Error = 0')
ax.set_xlabel('Error de predicción (minutos)')
ax.set_ylabel('Frecuencia')
ax.set_title('Distribución de errores de predicción')
ax.legend()

plt.tight_layout()
plt.show()

print(f"\nMedia de errores: {errores.mean():.2f} min (debe estar cerca de 0)")
print(f"Desviación estándar de errores: {errores.std():.2f} min")

## 13. Interpretabilidad del modelo

Para modelos basados en árboles (Random Forest, Gradient Boosting), podemos extraer la **importancia de features**.

In [ ]:
if best_model_name in ['RandomForest', 'GradientBoosting']:
    # Obtener importancias
    feature_importances = best_estimator.named_steps['model'].feature_importances_
    
    # Obtener nombres de features después del preprocesamiento
    feature_names = (numeric_features + 
                     list(best_estimator.named_steps['preprocessor']
                          .named_transformers_['cat']
                          .named_steps['onehot']
                          .get_feature_names_out(categorical_features)))
    
    # Crear DataFrame
    df_importance = pd.DataFrame({
        'feature': feature_names,
        'importance': feature_importances
    }).sort_values('importance', ascending=False).head(15)
    
    # Visualizar
    plt.figure(figsize=(10, 6))
    plt.barh(df_importance['feature'], df_importance['importance'])
    plt.xlabel('Importancia')
    plt.title(f'Top 15 Features más importantes ({best_model_name})')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    print("\n📊 Top 10 features más importantes:")
    print(df_importance.head(10).to_string(index=False))
else:
    print("\nℹ️ La interpretabilidad directa por importancia de features no está disponible para Ridge.")
    print("   Se pueden analizar los coeficientes del modelo.")
    
    # Para Ridge: mostrar coeficientes
    coefs = best_estimator.named_steps['model'].coef_
    feature_names = (numeric_features + 
                     list(best_estimator.named_steps['preprocessor']
                          .named_transformers_['cat']
                          .named_steps['onehot']
                          .get_feature_names_out(categorical_features)))
    
    df_coef = pd.DataFrame({
        'feature': feature_names,
        'coef': coefs
    }).sort_values(by='coef', key=abs, ascending=False).head(15)
    
    plt.figure(figsize=(10, 6))
    colors = ['red' if c < 0 else 'blue' for c in df_coef['coef']]
    plt.barh(df_coef['feature'], df_coef['coef'], color=colors)
    plt.xlabel('Coeficiente')
    plt.title('Top 15 Coeficientes más influyentes (Ridge)')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

## 14. Conclusiones y trabajo futuro

### Resumen de resultados

- **Problema redefinido:** Predicción de duración de partidos (regresión) ✅
- **Mejor modelo:** [se completa automáticamente según resultados]
- **Métricas finales en test:**
  - RMSE: [valor] minutos
  - MAE: [valor] minutos
  - R²: [valor]

### Diagnóstico

- [Overfitting / Underfitting / Balance adecuado] - se evalúa automáticamente
- Features más influyentes: `best_of`, ranking, superficie, ronda

### Limitaciones

1. **Variables no consideradas:** clima, estado físico de jugadores, momentum del partido
2. **Datos históricos limitados:** solo partidos disponibles en el dataset
3. **Outliers:** partidos muy largos (>300 min) o retiros pueden sesgar predicciones

### Trabajo futuro

1. **Ingeniería de features:**
   - Diferencia absoluta de ranking entre jugadores
   - Historial de enfrentamientos previos (H2H)
   - Racha reciente de victorias/derrotas
   - Interacciones entre superficie y estilo de juego (mano dominante)
   
2. **Modelos avanzados:**
   - XGBoost / LightGBM para mejor performance
   - Stacking de modelos (ensemble de segundo nivel)
   - Redes neuronales para capturar interacciones complejas
   
3. **Validación cruzada estratificada:**
   - Por superficie y por ronda del torneo simultáneamente
   - Time-series split para simular predicciones en tiempo real
   
4. **Análisis de errores:**
   - Identificar tipos de partidos con mayor error de predicción
   - Crear modelos especializados por segmento (Grand Slams vs otros)

---

**Entrega preparada para defensa oral:** El integrante Juan Ignacio Barranco puede explicar:
- ¿Por qué cambiamos el objetivo del proyecto?
- ¿Cómo funciona el pipeline de preprocesamiento?
- ¿Por qué elegimos este modelo sobre los otros?
- ¿Qué significan las métricas obtenidas?
- ¿Cómo diagnosticamos overfitting/underfitting?
- ¿Qué features son más importantes y por qué?